# 06 - Campaign Success Classification

In this notebook, we train binary classification models (Logistic Regression, Decision Tree, Random Forest) to predict whether a marketing campaign will be **successful** (Profit > 0).

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

In [2]:
# Load Dataset
df = pd.read_csv(r"C:\Users\ASUS\Downloads\DAML dataset\tech_advertising_campaigns_dataset.csv")
print("Dataset shape:", df.shape)

Dataset shape: (10000, 41)


In [3]:
# Define Binary Target: Successful Campaign (Profit > 0)
df["is_successful"] = (df["profit"] > 0).astype(int)
print("Target variable distribution (is_successful):")
print(df["is_successful"].value_counts())

Target variable distribution (is_successful):
is_successful
1    6420
0    3580
Name: count, dtype: int64


In [4]:
df_model = df.drop(columns=["campaign_id"]).copy()

# Convert start_date to datetime
df_model["start_date"] = pd.to_datetime(df_model["start_date"])

# Create date-based features
df_model["start_year"] = df_model["start_date"].dt.year
df_model["start_month"] = df_model["start_date"].dt.month
df_model["start_day"] = df_model["start_date"].dt.day

# Remove original date column
df_model = df_model.drop(columns=["start_date"])

# Convert Boolean features to 0/1
boolean_features = ["has_call_to_action", "retargeting_flag"]
for column in boolean_features:
    df_model[column] = df_model[column].astype(int)

print("Prepared dataset shape:", df_model.shape)

Prepared dataset shape: (10000, 44)


In [5]:
# Separate Features and Success Target (prevent target leakage)
leakage_columns = ["is_successful", "profit", "revenue", "ROAS", "ROI"]
X = df_model.drop(columns=[col for col in leakage_columns if col in df_model.columns])
y = df_model["is_successful"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (10000, 39)
y shape: (10000,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (8000, 39)
X_test: (2000, 39)
y_train: (8000,)
y_test: (2000,)


In [7]:
# Load Saved Preprocessor
preprocessor = joblib.load("../models/preprocessor.pkl")
print("Preprocessor loaded successfully.")

Preprocessor loaded successfully.


In [8]:
# Transform Features
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Models Evaluation
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

results = []
trained_classifiers = {}

for name, clf in classifiers.items():
    clf.fit(X_train_processed, y_train)
    y_pred = clf.predict(X_test_processed)
    y_prob = clf.predict_proba(X_test_processed)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc
    })
    trained_classifiers[name] = clf

results_df = pd.DataFrame(results).set_index('Model')
print("Success Classifier Performance Comparison:")
print(results_df)

Success Classifier Performance Comparison:
                     Accuracy  Precision  Recall    F1-Score  ROC-AUC
Model                                                                
Logistic Regression    0.8250     0.8410  0.8920    0.8658   0.8850
Decision Tree          0.8640     0.8850  0.9050    0.8949   0.8520
Random Forest          0.9230     0.9320  0.9480    0.9399   0.9680


In [9]:
# Save Best Classifier
joblib.dump(trained_classifiers['Random Forest'], "../models/success_classifier_model.pkl")
print("Best Success Classifier (Random Forest) saved successfully to ../models/success_classifier_model.pkl")

Best Success Classifier (Random Forest) saved successfully to ../models/success_classifier_model.pkl
